In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import glob
import random

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
DATA_DIR = "/content/drive/MyDrive/ai4i"

csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

print("Found CSV files:")
for path in csv_paths:
    print(path)

dfs = []

for i, path in enumerate(csv_paths):
    df_temp = pd.read_csv(path)
    df_temp["source_file"] = os.path.basename(path)
    df_temp["run_id"] = i
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Combined shape:", df.shape)
df.head()

Found CSV files:
/content/drive/MyDrive/ai4i/_world_navigation_metrics_20260507_043014.csv
/content/drive/MyDrive/ai4i/navigation_metrics_20260507_043014.csv
/content/drive/MyDrive/ai4i/navigation_metrics_20260507_045800.csv
Combined shape: (5259, 29)


,timestamp,total_distance,current_x,current_y,goal_id,goal_x,goal_y,commanded_speed,actual_speed,speed_error,...,obstacle_avoidance_efficiency,left_clearance,right_clearance,front_clearance,corridor_score,optimal_path_length,goal_reached,path_execution_time,source_file,run_id
0,2026-05-07T04:30:15,0.000000,0.000000,0.000000,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.196546,0.624742,1.927310,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
1,2026-05-07T04:30:16,0.000000,0.000000,0.000000,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.202477,0.627579,1.944675,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
2,2026-05-07T04:30:17,0.060407,0.059136,-0.012328,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.197795,0.615028,1.938316,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
3,2026-05-07T04:30:18,0.060407,0.059136,-0.012328,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.201949,0.595686,1.935018,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
4,2026-05-07T04:30:19,0.000000,0.059136,-0.012328,0,1.755,-0.525,0.0,0.0,0.0,...,0.0,1.196266,0.619699,1.925007,0.0,0.0,False,0.000000,_world_navigation_metrics_20260507_043014.csv,0


In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

print(df.columns.tolist())
df.head()

['timestamp', 'total_distance', 'current_x', 'current_y', 'goal_id', 'goal_x', 'goal_y', 'commanded_speed', 'actual_speed', 'speed_error', 'closest_obstacle_distance', 'mean_obstacle_distance', 'obstacle_density', 'goal_progress_rate', 'environment_complexity', 'navigation_status', 'battery_consumption', 'stuck_count', 'navigation_accuracydistance2goal', 'obstacle_avoidance_efficiency', 'left_clearance', 'right_clearance', 'front_clearance', 'corridor_score', 'optimal_path_length', 'goal_reached', 'path_execution_time', 'source_file', 'run_id']


,timestamp,total_distance,current_x,current_y,goal_id,goal_x,goal_y,commanded_speed,actual_speed,speed_error,...,obstacle_avoidance_efficiency,left_clearance,right_clearance,front_clearance,corridor_score,optimal_path_length,goal_reached,path_execution_time,source_file,run_id
0,2026-05-07T04:30:15,0.000000,0.000000,0.000000,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.196546,0.624742,1.927310,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
1,2026-05-07T04:30:16,0.000000,0.000000,0.000000,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.202477,0.627579,1.944675,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
2,2026-05-07T04:30:17,0.060407,0.059136,-0.012328,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.197795,0.615028,1.938316,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
3,2026-05-07T04:30:18,0.060407,0.059136,-0.012328,-1,0.000,0.000,0.0,0.0,0.0,...,0.0,1.201949,0.595686,1.935018,0.0,0.0,True,0.075701,_world_navigation_metrics_20260507_043014.csv,0
4,2026-05-07T04:30:19,0.000000,0.059136,-0.012328,0,1.755,-0.525,0.0,0.0,0.0,...,0.0,1.196266,0.619699,1.925007,0.0,0.0,False,0.000000,_world_navigation_metrics_20260507_043014.csv,0


In [5]:
feature_cols = [
    "total_distance",
    "current_x",
    "current_y",
    "goal_x",
    "goal_y",
    "commanded_speed",
    "actual_speed",
    "speed_error",
    "closest_obstacle_distance",
    "mean_obstacle_distance",
    "obstacle_density",
    "goal_progress_rate",
    "environment_complexity",
    "battery_consumption",
    "navigation_accuracydistance2goal",
    "obstacle_avoidance_efficiency",
    "left_clearance",
    "right_clearance",
    "front_clearance",
    "corridor_score",
    "optimal_path_length",
    "path_execution_time",
]

target_col = "stuck_count"

missing = [c for c in feature_cols + [target_col] if c not in df.columns]
print("Missing columns:", missing)

Missing columns: []


In [6]:
df["slow_progress"] = (df["goal_progress_rate"] < 0.005).astype(int)
df["close_obstacle"] = (df["closest_obstacle_distance"] < 0.45).astype(int)
df["high_complexity"] = (df["environment_complexity"] > 0.8).astype(int)
df["narrow"] = (df["corridor_score"] > 0.5).astype(int)
df["bad_efficiency"] = (df["obstacle_avoidance_efficiency"] > 1.8).astype(int)

df["navigation_risk"] = (
    df["slow_progress"] +
    df["close_obstacle"] +
    df["high_complexity"] +
    df["narrow"] +
    df["bad_efficiency"]
) / 5.0

df["risk_class"] = (df["navigation_risk"] >= 0.6).astype(int)

print(df["risk_class"].value_counts())

risk_class
0    4103
1    1156
Name: count, dtype: int64


In [7]:
df = df.sort_values(["run_id", "goal_id"]).reset_index(drop=True)

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
df[feature_cols] = df[feature_cols].fillna(0.0)

df[feature_cols].head()

,total_distance,current_x,current_y,goal_x,goal_y,commanded_speed,actual_speed,speed_error,closest_obstacle_distance,mean_obstacle_distance,...,environment_complexity,battery_consumption,navigation_accuracydistance2goal,obstacle_avoidance_efficiency,left_clearance,right_clearance,front_clearance,corridor_score,optimal_path_length,path_execution_time
0,0.000000,0.000000,0.000000,0.000,0.000,0.0,0.0,0.0,0.511738,1.364291,...,0.321245,0.00000,0.000000,0.0,1.196546,0.624742,1.927310,0.0,0.0,0.075701
1,0.000000,0.000000,0.000000,0.000,0.000,0.0,0.0,0.0,0.504501,1.362655,...,0.321631,0.00000,0.000000,0.0,1.202477,0.627579,1.944675,0.0,0.0,0.075701
2,0.060407,0.059136,-0.012328,0.000,0.000,0.0,0.0,0.0,0.509547,1.363347,...,0.321467,0.00302,0.060407,0.0,1.197795,0.615028,1.938316,0.0,0.0,0.075701
3,0.060407,0.059136,-0.012328,0.000,0.000,0.0,0.0,0.0,0.517612,1.363635,...,0.323663,0.00302,0.060407,0.0,1.201949,0.595686,1.935018,0.0,0.0,0.075701
4,0.000000,0.059136,-0.012328,1.755,-0.525,0.0,0.0,0.0,0.513151,1.364184,...,0.319008,0.00302,0.060407,0.0,1.196266,0.619699,1.925007,0.0,0.0,0.000000


In [8]:
class NavigationSequenceDataset(Dataset):
    def __init__(self, dataframe, feature_cols, label_col, window_size=8):
        self.X = []
        self.y = []

        grouped = dataframe.groupby(["run_id", "goal_id"])

        for _, group in grouped:
            group = group.reset_index(drop=True)

            if len(group) < window_size:
                continue

            features = group[feature_cols].values.astype(np.float32)
            labels = group[label_col].values.astype(np.int64)

            for i in range(len(group) - window_size + 1):
                x_window = features[i:i + window_size]
                y_label = labels[i + window_size - 1]

                self.X.append(x_window)
                self.y.append(y_label)

        self.X = np.array(self.X, dtype=np.float32)
        self.y = np.array(self.y, dtype=np.int64)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])

In [9]:
window_size = 5

dataset_raw = NavigationSequenceDataset(
    df,
    feature_cols,
    label_col="risk_class",
    window_size=window_size
)

print("Total sequences:", len(dataset_raw))
print("X shape:", dataset_raw.X.shape)
print("y distribution:", np.bincount(dataset_raw.y))

Total sequences: 4664
X shape: (4664, 5, 22)
y distribution: [3810  854]


In [10]:
X = dataset_raw.X
y = dataset_raw.y

num_samples, seq_len, num_features = X.shape

X_flat = X.reshape(-1, num_features)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)

X_scaled = X_scaled.reshape(num_samples, seq_len, num_features)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if len(np.unique(y)) > 1 else None
)

class TensorSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = TensorSequenceDataset(X_train, y_train)
val_dataset = TensorSequenceDataset(X_val, y_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Input features:", num_features)

Train: 3731
Val: 933
Input features: 22


In [11]:
class NavigationLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        lstm_out, (hidden, cell) = self.lstm(x)

        last_hidden = hidden[-1]

        logits = self.classifier(last_hidden)

        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NavigationLSTM(
    input_size=num_features,
    hidden_size=64,
    num_layers=2,
    dropout=0.2
).to(device)

model

NavigationLSTM(
  (lstm): LSTM(22, 64, num_layers=2, batch_first=True, dropout=0.2)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=32, out_features=2, bias=True)
  )
)

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# class weights for imbalance
class_counts = np.bincount(y_train)

if len(class_counts) < 2:
    class_weights = torch.tensor([1.0, 1.0], dtype=torch.float32).to(device)
else:
    total = class_counts.sum()
    class_weights = total / (2.0 * class_counts)
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

num_epochs = 25


def train_one_epoch(model, loader):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(y_batch.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    f1 = f1_score(all_targets, all_preds, zero_division=0)
    acc = accuracy_score(all_targets, all_preds)

    return avg_loss, acc, f1


def evaluate(model, loader):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(y_batch.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(all_targets, all_preds)
    prec = precision_score(all_targets, all_preds, zero_division=0)
    rec = recall_score(all_targets, all_preds, zero_division=0)
    f1 = f1_score(all_targets, all_preds, zero_division=0)

    return avg_loss, acc, prec, rec, f1, all_targets, all_preds


best_val_f1 = -1
best_state = None

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader)
    val_loss, val_acc, val_prec, val_rec, val_f1, y_true, y_pred = evaluate(model, val_loader)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} train_f1={train_f1:.3f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} "
        f"val_prec={val_prec:.3f} val_rec={val_rec:.3f} val_f1={val_f1:.3f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict().copy()

if best_state is not None:
    model.load_state_dict(best_state)

print("Best validation F1:", best_val_f1)
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Epoch 01 | train_loss=0.3375 train_acc=0.786 train_f1=0.616 | val_loss=0.2002 val_acc=0.883 val_prec=0.620 val_rec=0.936 val_f1=0.746
Epoch 02 | train_loss=0.1855 train_acc=0.913 train_f1=0.798 | val_loss=0.1253 val_acc=0.935 val_prec=0.746 val_rec=0.977 val_f1=0.846
Epoch 03 | train_loss=0.1221 train_acc=0.947 train_f1=0.870 | val_loss=0.0884 val_acc=0.976 val_prec=0.931 val_rec=0.942 val_f1=0.936
Epoch 04 | train_loss=0.0797 train_acc=0.965 train_f1=0.911 | val_loss=0.0805 val_acc=0.974 val_prec=0.915 val_rec=0.947 val_f1=0.931
Epoch 05 | train_loss=0.0631 train_acc=0.976 train_f1=0.937 | val_loss=0.0769 val_acc=0.950 val_prec=0.790 val_rec=0.988 val_f1=0.878
Epoch 06 | train_loss=0.0474 train_acc=0.982 train_f1=0.951 | val_loss=0.0842 val_acc=0.982 val_prec=0.958 val_rec=0.942 val_f1=0.950
Epoch 07 | train_loss=0.0485 train_acc=0.981 train_f1=0.949 | val_loss=0.0507 val_acc=0.977 val_prec=0.903 val_rec=0.982 val_f1=0.941
Epoch 08 | train_loss=0.0356 train_acc=0.984 train_f1=0.958 | 

In [14]:
SAVE_DIR = "/content/drive/MyDrive/ai4i/models"

os.makedirs(SAVE_DIR, exist_ok=True)

MODEL_PATH = os.path.join(SAVE_DIR, "navigation_lstm_model2.pth")
SCALER_PATH = os.path.join(SAVE_DIR, "navigation_scaler2.pkl")
FEATURES_PATH = os.path.join(SAVE_DIR, "feature_columns2.pkl")

torch.save(model.state_dict(), MODEL_PATH)

import joblib

joblib.dump(scaler, SCALER_PATH)
joblib.dump(feature_cols, FEATURES_PATH)

print("Model saved to:")
print(MODEL_PATH)

print("\nScaler saved to:")
print(SCALER_PATH)

print("\nFeature columns saved to:")
print(FEATURES_PATH)

Model saved to:
/content/drive/MyDrive/ai4i/models/navigation_lstm_model2.pth

Scaler saved to:
/content/drive/MyDrive/ai4i/models/navigation_scaler2.pkl

Feature columns saved to:
/content/drive/MyDrive/ai4i/models/feature_columns2.pkl
